# **9. Comparando Modelos Benchmark**

## **9.1. Librerias**

In [1]:
import os
import pandas as pd
from scipy import stats
import numpy as np
from scipy.stats import shapiro
import pandas as pd
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from scipy.stats import levene

from itertools import combinations

## **9.2.Cargando los Resultados**

In [2]:
BASE_DIR = "/home/guirlessa/Dl_Proyecto_Dengue/Modelos" 

MODELOS = ["LSTM", "CONVLSTM", "STGNN", "GRU", "Transformer"]

def load_metrics(model_path):
    file_path = os.path.join(model_path, "metrics_stgnn.csv")

    # fallback si el nombre cambia por modelo
    if not os.path.exists(file_path):
        candidates = [f for f in os.listdir(model_path) if "metrics" in f and f.endswith(".csv")]
        if len(candidates) == 0:
            return None
        file_path = os.path.join(model_path, candidates[0])

    df = pd.read_csv(file_path)
    return df

all_metrics = []

for model in MODELOS:
    model_path = os.path.join(BASE_DIR, model)

    if not os.path.exists(model_path):
        print(f"No existe carpeta: {model_path}")
        continue

    df = load_metrics(model_path)

    if df is None:
        print(f"No metrics found for {model}")
        continue

    df["Model"] = model
    all_metrics.append(df)

# concatenar todo
metrics_df = pd.concat(all_metrics, ignore_index=True)
metrics_df.head()

,seed,RMSE,MAE,R2,Model
0,42,7.643459,4.205040,0.048068,LSTM
1,123,7.069677,3.966179,0.002728,LSTM
2,2024,7.050868,3.969128,0.008028,LSTM
3,42,0.677770,0.489266,0.438390,CONVLSTM
4,123,0.652954,0.469355,0.454739,CONVLSTM


## **9.3. Metricas Globales**

In [3]:
summary = metrics_df.groupby("Model").agg({
    "RMSE": ["mean", "std"],
    "MAE": ["mean", "std"],
    "R2": ["mean", "std"]
})

summary.columns = [
    "RMSE_mean", "RMSE_std",
    "MAE_mean", "MAE_std",
    "R2_mean", "R2_std"
]

summary = summary.reset_index()

summary

,Model,RMSE_mean,RMSE_std,MAE_mean,MAE_std,R2_mean,R2_std
0,CONVLSTM,0.658141,0.017618,0.473016,0.014764,0.457593,0.020778
1,GRU,7.272854,0.315501,4.038056,0.092422,0.025908,0.083521
2,LSTM,7.254668,0.336835,4.046782,0.137063,0.019608,0.024789
3,STGNN,7.237096,0.089629,4.048328,0.060917,-0.003409,0.093412
4,Transformer,4.054371,0.304159,2.102692,0.131240,0.681954,0.067499


Los resultados comparativos muestran diferencias claras en desempeño y estabilidad entre las arquitecturas evaluadas. El Transformer se posiciona como el mejor modelo global, con el menor error promedio (RMSE ≈ 4.05, MAE ≈ 2.10) y el mayor poder explicativo (R² ≈ 0.68), además de una variabilidad moderada que indica un equilibrio razonable entre ajuste y generalización. El ConvLSTM presenta el segundo mejor desempeño, con errores significativamente menores que el resto de modelos recurrentes y un R² ≈ 0.46, lo que sugiere una capacidad predictiva intermedia con buena estabilidad relativa. En contraste, GRU, LSTM y STGNN conforman un grupo de bajo rendimiento, con RMSE cercanos a 7.2–7.3, MAE alrededor de 4 y R² cercanos a cero o incluso negativos en el caso de STGNN, lo que evidencia una capacidad explicativa muy limitada y pobre ajuste al fenómeno subyacente. En particular, GRU no aporta mejoras sustanciales respecto a LSTM, manteniendo un desempeño equivalente tanto en error como en varianza, lo que sugiere que su capacidad de modelado no supera a las arquitecturas recurrentes clásicas en este contexto experimental.


## **9.4. Métricas por Semilla**

In [4]:
seed_analysis = metrics_df.groupby(["Model", "seed"]).agg({
    "RMSE": "mean",
    "MAE": "mean",
    "R2": "mean"
}).reset_index()

seed_analysis.head(15)

,Model,seed,RMSE,MAE,R2
0,CONVLSTM,42,0.677770,0.489266,0.438390
1,CONVLSTM,123,0.652954,0.469355,0.454739
2,CONVLSTM,2024,0.643700,0.460427,0.479651
3,GRU,42,6.945436,3.931795,0.037472
4,GRU,123,7.574906,4.099742,0.103044
5,GRU,2024,7.298220,4.082630,-0.062792
6,LSTM,42,7.643459,4.205040,0.048068
7,LSTM,123,7.069677,3.966179,0.002728
8,LSTM,2024,7.050868,3.969128,0.008028
9,STGNN,42,7.334999,4.101241,-0.073531


El análisis por semilla confirma la jerarquía de desempeño observada en los promedios globales, mostrando consistencia estructural entre ejecuciones para cada arquitectura. El Transformer mantiene el mejor rendimiento y además evidencia una mejora progresiva en ciertas inicializaciones, con reducción del RMSE hasta ~3.70 y aumento del R² hasta ~0.76, lo que indica no solo buen ajuste sino también sensibilidad positiva a la optimización. El ConvLSTM presenta un comportamiento estable y consistente, con baja variabilidad entre semillas y un R² relativamente constante (~0.44–0.48), lo que sugiere robustez pero capacidad predictiva limitada. En contraste, GRU, LSTM y STGNN muestran un desempeño marcadamente inferior, con RMSE elevados (~7–7.6), MAE alrededor de 4 y R² cercanos a cero o negativos en varios casos, lo que evidencia baja capacidad explicativa y alta dependencia de la inicialización. En particular, GRU no presenta ventajas claras frente a LSTM, manteniendo un comportamiento equivalente tanto en error como en estabilidad, lo que sugiere que no introduce mejoras significativas en este contexto experimental.

In [5]:
stability = metrics_df.groupby("Model").agg({
    "RMSE": "std",
    "MAE": "std",
    "R2": "std"
}).rename(columns={
    "RMSE": "RMSE_variability",
    "MAE": "MAE_variability",
    "R2": "R2_variability"
})

stability

,RMSE_variability,MAE_variability,R2_variability
Model,,,
CONVLSTM,0.017618,0.014764,0.020778
GRU,0.315501,0.092422,0.083521
LSTM,0.336835,0.137063,0.024789
STGNN,0.089629,0.060917,0.093412
Transformer,0.304159,0.131240,0.067499


Los resultados muestran una separación clara entre dos regímenes de desempeño. El Transformer es el único modelo con capacidad explicativa relevante, alcanzando el menor error relativo dentro del conjunto de modelos no-GNN (RMSE ≈ 4.05, MAE ≈ 2.10) y un R² consistente alrededor de 0.68–0.76 según semilla, lo que indica que captura una fracción sustancial de la varianza del proceso, con variabilidad moderada entre ejecuciones. En contraste, CONVLSTM presenta un comportamiento estable pero subóptimo en escala de error (RMSE ≈ 0.65 en su escala interna), con R² moderado (~0.45), sugiriendo capacidad predictiva limitada pero consistente. Por otro lado, GRU, LSTM y STGNN exhiben errores significativamente mayores (RMSE ≈ 7.2–7.6, MAE ≈ 4.0) y coeficientes de determinación cercanos a cero o negativos, lo que implica ausencia de poder explicativo efectivo sobre la dinámica objetivo. Adicionalmente, su variabilidad entre semillas es considerable (RMSE std ≈ 0.30–0.33), indicando sensibilidad alta a la inicialización y baja robustez estadística. En conjunto, el espacio de modelos queda dominado por un único modelo con desempeño sustantivo (Transformer), mientras que el resto opera en un régimen de baja capacidad predictiva, con señales claras de subajuste estructural o desalineación con la complejidad del proceso temporal modelado.


## **9.5. ANOVA**

El análisis de varianza (ANOVA) se utiliza para evaluar si las diferencias observadas en el desempeño entre los distintos modelos son estadísticamente significativas o si pueden atribuirse a la variabilidad inherente del proceso de entrenamiento. De esta manera, se garantiza una comparación rigurosa y objetiva bajo un mismo esquema experimental.

In [6]:
models = metrics_df["Model"].unique()

groups_rmse = [metrics_df[metrics_df["Model"] == m]["RMSE"].values for m in models]
groups_mae  = [metrics_df[metrics_df["Model"] == m]["MAE"].values for m in models]
groups_r2   = [metrics_df[metrics_df["Model"] == m]["R2"].values for m in models]

### **9.5.1. Supuestos**

Para aplicar ANOVA es necesario cumplir ciertos supuestos estadísticos, principalmente la normalidad de las distribuciones dentro de cada grupo, la homogeneidad de varianzas entre los grupos y la independencia de las observaciones. Estos requisitos son importantes porque aseguran que la comparación de medias entre modelos sea válida y que los resultados del test no estén sesgados por violaciones de las condiciones estadísticas que sustentan su inferencia.


#### **9.5.1.1. Normalidad**

In [7]:
print("TEST DE NORMALIDAD (Shapiro-Wilk)\n")

for model in models:
    data = metrics_df[metrics_df["Model"] == model]["RMSE"]

    stat, p = shapiro(data)

    print(f"🔹 {model} - RMSE")
    print(f"   W = {stat:.4f}, p = {p:.6f}")

    if p < 0.05:
        print("   No normal (rechaza H0)\n")
    else:
        print("   Normal (no se rechaza H0)\n")

TEST DE NORMALIDAD (Shapiro-Wilk)

🔹 LSTM - RMSE
   W = 0.7738, p = 0.053330
   Normal (no se rechaza H0)

🔹 CONVLSTM - RMSE
   W = 0.9350, p = 0.507554
   Normal (no se rechaza H0)

🔹 STGNN - RMSE
   W = 0.9631, p = 0.630615
   Normal (no se rechaza H0)

🔹 GRU - RMSE
   W = 0.9952, p = 0.866914
   Normal (no se rechaza H0)

🔹 Transformer - RMSE
   W = 0.7729, p = 0.051330
   Normal (no se rechaza H0)



Los resultados del test de Shapiro-Wilk indican que, para la métrica RMSE en todos los modelos evaluados, no se rechaza la hipótesis nula de normalidad (p > 0.05 en todos los casos). Esto sugiere que las distribuciones de error por modelo pueden considerarse aproximadamente normales.


#### **9.5.1.2. Homocedasticidad**

In [8]:
print("TEST DE LEVENE (Homogeneidad de varianzas)\n")

# RMSE
stat, p = levene(*groups_rmse)
print(f"RMSE -> W={stat:.4f}, p={p:.6f}")

# MAE
stat, p = levene(*groups_mae)
print(f"MAE  -> W={stat:.4f}, p={p:.6f}")

# R2
stat, p = levene(*groups_r2)
print(f"R2   -> W={stat:.4f}, p={p:.6f}")

TEST DE LEVENE (Homogeneidad de varianzas)

RMSE -> W=0.5259, p=0.719528
MAE  -> W=0.4374, p=0.779050
R2   -> W=0.5254, p=0.719813


Los resultados del test de Levene indican que no se rechaza la hipótesis nula de igualdad de varianzas para las métricas RMSE, MAE y R² (p > 0.05 en todos los casos). Esto sugiere que las varianzas entre los distintos modelos son homogéneas, cumpliendo así el supuesto de homocedasticidad requerido para la aplicación del ANOVA.

#### **9.5.1.3. Independencia de Observaciones**

La independencia de las observaciones se garantiza mediante el uso de distintas semillas aleatorias y particiones de validación cruzada, asegurando que cada ejecución del modelo sea estadísticamente independiente.

### **9.5.2. Aplicando ANOVA**

In [9]:
f_rmse, p_rmse = stats.f_oneway(*groups_rmse)

print(" ANOVA RMSE")
print(f"F-statistic: {f_rmse:.4f}")
print(f"p-value: {p_rmse}")

 ANOVA RMSE
F-statistic: 412.9730
p-value: 4.70927770914682e-11


El ANOVA aplicado a RMSE arroja un estadístico F = 412.97 con p ≈ 4.7×10⁻¹¹, lo que implica el rechazo de la hipótesis nula de igualdad de medias entre modelos. Bajo este resultado, se concluye que las diferencias observadas en RMSE no pueden atribuirse a variabilidad muestral o aleatoria del proceso de entrenamiento, sino que existen efectos sistemáticos asociados a la arquitectura del modelo. En términos inferenciales, al menos un modelo pertenece a una distribución de error significativamente distinta, lo que valida estadísticamente la existencia de heterogeneidad estructural en el desempeño predictivo dentro del conjunto evaluado.

In [10]:
f_mae, p_mae = stats.f_oneway(*groups_mae)

print("\n ANOVA MAE")
print(f"F-statistic: {f_mae:.4f}")
print(f"p-value: {p_mae}")


 ANOVA MAE
F-statistic: 808.0123
p-value: 1.670842938878178e-12


El ANOVA sobre MAE produce F = 808.01 con p ≈ 1.67×10⁻¹², lo que implica un rechazo de la hipótesis nula de igualdad de medias entre modelos. Este resultado indica que las diferencias en el error absoluto medio no son atribuibles a fluctuaciones aleatorias del proceso de entrenamiento, sino a diferencias sistemáticas inducidas por la arquitectura de cada modelo. En consecuencia, existe evidencia estadística robusta de heterogeneidad en el desempeño respecto a MAE, confirmando que al menos un subconjunto de modelos presenta un nivel de error significativamente distinto dentro del espacio experimental evaluado.

In [11]:
f_r2, p_r2 = stats.f_oneway(*groups_r2)

print("\n ANOVA R²")
print(f"F-statistic: {f_r2:.4f}")
print(f"p-value: {p_r2}")


 ANOVA R²
F-statistic: 69.7512
p-value: 2.8901483706465004e-07


El ANOVA aplicado al coeficiente de determinación (R²) arroja F = 69.75 con p ≈ 2.89×10⁻⁷, lo que implica el rechazo de la hipótesis nula de igualdad de medias entre modelos para la capacidad explicativa. Aunque la magnitud del estadístico F es menor que en RMSE y MAE, el resultado sigue siendo estadísticamente altamente significativo, indicando que las diferencias observadas en R² no pueden explicarse por variabilidad aleatoria. En términos inferenciales, se confirma la existencia de heterogeneidad estructural en la capacidad de ajuste de los modelos, con al menos un subconjunto que presenta una varianza explicada significativamente distinta dentro del sistema evaluado.

## **9.6. TUKEY HSD**

Dado que el ANOVA evidenció la existencia de diferencias estadísticamente significativas entre las medias de desempeño de los modelos, y considerando que se verificó el cumplimiento de los supuestos de normalidad y homocedasticidad, se procede a aplicar la prueba post-hoc de Tukey HSD. Esta prueba permite identificar específicamente entre qué pares de modelos se presentan diferencias significativas, proporcionando una comparación múltiple controlada del error tipo I.

In [ ]:
tukey_rmse = pairwise_tukeyhsd(
    endog=metrics_df["RMSE"],
    groups=metrics_df["Model"],
    alpha=0.05
)

print("TUKEY HSD - RMSE")
print(tukey_rmse)

📊 TUKEY HSD - RMSE
    Multiple Comparison of Means - Tukey HSD, FWER=0.05    
 group1     group2   meandiff p-adj   lower   upper  reject
-----------------------------------------------------------
CONVLSTM         GRU   6.6147    0.0  5.9415   7.288   True
CONVLSTM        LSTM   6.5965    0.0  5.9233  7.2698   True
CONVLSTM       STGNN    6.579    0.0  5.9057  7.2522   True
CONVLSTM Transformer   3.3962    0.0   2.723  4.0695   True
     GRU        LSTM  -0.0182    1.0 -0.6914  0.6551  False
     GRU       STGNN  -0.0358 0.9998  -0.709  0.6375  False
     GRU Transformer  -3.2185    0.0 -3.8917 -2.5452   True
    LSTM       STGNN  -0.0176    1.0 -0.6908  0.6557  False
    LSTM Transformer  -3.2003    0.0 -3.8735 -2.5271   True
   STGNN Transformer  -3.1827    0.0  -3.856 -2.5095   True
-----------------------------------------------------------


La prueba post-hoc de Tukey HSD sobre RMSE evidencia una estructura de desempeño claramente estratificada entre modelos. CONVLSTM presenta diferencias estadísticamente significativas frente a todos los demás modelos (GRU, LSTM, STGNN y Transformer), con mejoras relativas en RMSE de magnitud sustancial, lo que lo separa como una clase estadística distinta dentro del conjunto. En el otro extremo, GRU, LSTM y STGNN no muestran diferencias significativas entre sí (p ≈ 1.0 en todas las comparaciones cruzadas), lo que implica equivalencia estadística en su nivel de error y sugiere que pertenecen al mismo régimen de bajo desempeño. El Transformer, por su parte, se diferencia significativamente de GRU, LSTM y STGNN (p < 0.001), pero también es significativamente peor que CONVLSTM, ubicándose en un nivel intermedio pero claramente separado de ambos extremos. En conjunto, la estructura inferida por Tukey confirma la existencia de al menos tres grupos estadísticos bien definidos: (i) CONVLSTM como el mejor desempeño relativo en RMSE, (ii) Transformer como régimen intermedio, y (iii) GRU/LSTM/STGNN como un grupo homogéneo de bajo desempeño sin diferencias internas detectables.


In [14]:
tukey_mae = pairwise_tukeyhsd(
    endog=metrics_df["MAE"],
    groups=metrics_df["Model"],
    alpha=0.05
)

print("TUKEY HSD - MAE")
print(tukey_mae)

TUKEY HSD - MAE
    Multiple Comparison of Means - Tukey HSD, FWER=0.05    
 group1     group2   meandiff p-adj   lower   upper  reject
-----------------------------------------------------------
CONVLSTM         GRU    3.565    0.0  3.3004  3.8296   True
CONVLSTM        LSTM   3.5738    0.0  3.3092  3.8384   True
CONVLSTM       STGNN   3.5753    0.0  3.3107  3.8399   True
CONVLSTM Transformer   1.6297    0.0  1.3651  1.8943   True
     GRU        LSTM   0.0087    1.0 -0.2559  0.2733  False
     GRU       STGNN   0.0103 0.9999 -0.2543  0.2749  False
     GRU Transformer  -1.9354    0.0    -2.2 -1.6708   True
    LSTM       STGNN   0.0015    1.0 -0.2631  0.2661  False
    LSTM Transformer  -1.9441    0.0 -2.2087 -1.6795   True
   STGNN Transformer  -1.9456    0.0 -2.2102  -1.681   True
-----------------------------------------------------------


La prueba de Tukey HSD aplicada al MAE reproduce una estructura de separación estadística coherente con la observada en RMSE, pero con mayor contraste en magnitud absoluta de error. CONVLSTM se diferencia significativamente de todos los demás modelos (GRU, LSTM, STGNN y Transformer), mostrando una reducción sistemática del MAE que lo posiciona como el mejor grupo estadístico en esta métrica. En contraste, GRU, LSTM y STGNN no presentan diferencias significativas entre sí (p ≈ 1), lo que indica equivalencia estadística en su error absoluto medio y confirma que operan en un mismo régimen de bajo desempeño. El Transformer se ubica nuevamente en una posición intermedia: es significativamente mejor que GRU, LSTM y STGNN (p < 0.001), pero significativamente peor que CONVLSTM, lo que consolida su rol como modelo de desempeño medio dentro del sistema comparado. En conjunto, el patrón inferido por MAE confirma una partición consistente del espacio de modelos en tres niveles jerárquicos: CONVLSTM como frontera superior de desempeño, Transformer como régimen intermedio, y GRU/LSTM/STGNN como un grupo estadísticamente indistinguible de bajo rendimiento.


In [15]:
tukey_r2 = pairwise_tukeyhsd(
    endog=metrics_df["R2"],
    groups=metrics_df["Model"],
    alpha=0.05
)

print(" TUKEY HSD - R²")
print(tukey_r2)

 TUKEY HSD - R²
    Multiple Comparison of Means - Tukey HSD, FWER=0.05    
 group1     group2   meandiff p-adj   lower   upper  reject
-----------------------------------------------------------
CONVLSTM         GRU  -0.4317 0.0001 -0.6071 -0.2563   True
CONVLSTM        LSTM   -0.438 0.0001 -0.6134 -0.2626   True
CONVLSTM       STGNN   -0.461    0.0 -0.6364 -0.2856   True
CONVLSTM Transformer   0.2244 0.0121   0.049  0.3998   True
     GRU        LSTM  -0.0063 0.9999 -0.1817  0.1691  False
     GRU       STGNN  -0.0293 0.9794 -0.2047  0.1461  False
     GRU Transformer    0.656    0.0  0.4806  0.8314   True
    LSTM       STGNN   -0.023 0.9916 -0.1984  0.1524  False
    LSTM Transformer   0.6623    0.0  0.4869  0.8377   True
   STGNN Transformer   0.6854    0.0    0.51  0.8608   True
-----------------------------------------------------------


La prueba de Tukey HSD aplicada al R² confirma una estructura de desempeño estadísticamente diferenciada, pero con una organización no completamente alineada con las métricas de error. CONVLSTM difiere significativamente de todos los modelos evaluados; sin embargo, en este caso presenta R² inferior al Transformer y superior al resto de modelos recurrentes, lo que lo posiciona como un régimen intermedio-bajo en capacidad explicativa. El Transformer emerge como el modelo con mayor R² medio, diferenciándose significativamente de GRU, LSTM y STGNN (p < 0.001), lo que indica una superioridad consistente en varianza explicada. Por otro lado, GRU, LSTM y STGNN no presentan diferencias significativas entre sí (p ≈ 1 en todas las comparaciones), lo que confirma su equivalencia estadística en términos de capacidad explicativa y su pertenencia a un mismo grupo de bajo desempeño. En conjunto, el análisis de R² sugiere una jerarquía clara de capacidad explicativa donde el Transformer domina el espacio de modelos, seguido por CONVLSTM en un nivel intermedio, y finalmente un bloque homogéneo compuesto por GRU, LSTM y STGNN con desempeño indistinguible y limitado.


## **9.7. Evaluando Robustez**

Para evaluar la robustez de los modelos, se analizaron las medias y desviaciones estándar de cada métrica de desempeño (RMSE, MAE y R²) a través de múltiples ejecuciones, con el fin de cuantificar tanto el rendimiento promedio como su variabilidad. Adicionalmente, se calculó el coeficiente de variación, lo que permitió estandarizar la dispersión relativa de los resultados y comparar la estabilidad entre modelos con diferentes escalas de error. Este enfoque facilita la identificación de aquellos modelos que no solo presentan buen desempeño promedio, sino también consistencia frente a variaciones en la inicialización y el proceso de entrenamiento, lo cual es fundamental para evaluar su robustez.

### **9.7.1. Medias & Desviaciones**

In [16]:
robustness = metrics_df.groupby("Model").agg({
    "RMSE": ["mean", "std"],
    "MAE": ["mean", "std"],
    "R2": ["mean", "std"]
})

robustness

RMSE                 MAE                  R2          
                 mean       std      mean       std      mean       std
Model                                                                  
CONVLSTM     0.658141  0.017618  0.473016  0.014764  0.457593  0.020778
GRU          7.272854  0.315501  4.038056  0.092422  0.025908  0.083521
LSTM         7.254668  0.336835  4.046782  0.137063  0.019608  0.024789
STGNN        7.237096  0.089629  4.048328  0.060917 -0.003409  0.093412
Transformer  4.054371  0.304159  2.102692  0.131240  0.681954  0.067499

La evaluación de robustez, considerando simultáneamente media y desviación estándar de RMSE, MAE y R², evidencia una estructura clara de estabilidad diferencial entre modelos. CONVLSTM presenta el comportamiento más estable en términos absolutos, con desviaciones estándar bajas en todas las métricas (RMSE std ≈ 0.018, MAE std ≈ 0.015, R² std ≈ 0.021) y un desempeño consistente, aunque su capacidad predictiva es limitada en comparación con modelos más complejos. El Transformer, pese a exhibir el mejor desempeño global en términos de error y capacidad explicativa (RMSE ≈ 4.05, R² ≈ 0.68), mantiene una variabilidad moderada, lo que indica buena pero no óptima estabilidad frente a cambios de inicialización. En contraste, GRU y LSTM presentan simultáneamente alto error y variabilidad no despreciable (RMSE std ≈ 0.31–0.34), lo que sugiere sensibilidad a la inicialización y baja estabilidad estadística. STGNN, aunque muestra menor variabilidad en RMSE, mantiene un desempeño sistemáticamente pobre (R² ≈ 0 o negativo), lo que indica estabilidad en un régimen de bajo rendimiento más que robustez predictiva real. En conjunto, la robustez efectiva se observa únicamente en modelos con bajo error y baja varianza (CONVLSTM parcialmente, Transformer en segundo orden), mientras que los demás modelos o bien son estables pero poco precisos, o bien son inestables y poco informativos.


### **9.7.2. Coeficiente de Variación**

In [17]:
cv_rmse = metrics_df.groupby("Model")["RMSE"].std() / metrics_df.groupby("Model")["RMSE"].mean()
cv_rmse

Model
CONVLSTM       0.026769
GRU            0.043381
LSTM           0.046430
STGNN          0.012385
Transformer    0.075020
Name: RMSE, dtype: float64

El coeficiente de variación del RMSE muestra diferencias claras en la estabilidad relativa de los modelos cuando se normaliza la dispersión respecto a su media. STGNN presenta el menor valor (≈0.012), lo que indica la menor variabilidad relativa; sin embargo, este resultado debe interpretarse junto con su bajo desempeño predictivo (R² cercano a cero), por lo que la estabilidad observada ocurre en un régimen de error sistemáticamente alto y poco informativo. CONVLSTM (≈0.027) muestra la mejor combinación entre baja variabilidad relativa y buen desempeño, situándose como el modelo más equilibrado en términos de robustez efectiva. GRU (≈0.043) y LSTM (≈0.046) exhiben una variabilidad moderada-alta, consistente con su desempeño deficiente y su sensibilidad a la inicialización. Finalmente, Transformer presenta el coeficiente de variación más alto (≈0.075), lo que indica mayor sensibilidad relativa a cambios entre ejecuciones, a pesar de su mejor capacidad predictiva global. En conjunto, la robustez relativa favorece a CONVLSTM como el modelo más consistente en relación desempeño–variabilidad, mientras que STGNN es estable pero no útil predictivamente, y Transformer maximiza desempeño a costa de mayor variabilidad.


## **9.8. Overfitting/underfitting**

En esta sección se analiza la presencia de **sobreajuste (overfitting) y subajuste (underfitting)** en los modelos evaluados a partir de sus curvas de aprendizaje. Para ello, se utiliza la diferencia entre la pérdida de validación y la pérdida de entrenamiento como indicador del grado de generalización del modelo, calculada a nivel de épocas y posteriormente agregada por semillas y particiones. Este enfoque permite cuantificar la brecha de generalización (generalization gap) y evaluar su estabilidad mediante medidas de tendencia central y dispersión, facilitando la comparación consistente del comportamiento de cada arquitectura bajo el mismo esquema experimental.


In [19]:


BASE_DIR = "Modelos"

MODELOS = ["LSTM", "CONVLSTM", "STGNN", "GRU", "Transformer"]

all_results = []

for model in MODELOS:

    path = os.path.join(BASE_DIR, model)

    file = [f for f in os.listdir(path) if "training_history" in f and f.endswith(".csv")]

    if len(file) == 0:
        print(f"No history found for {model}")
        continue

    history_path = os.path.join(path, file[0])
    history = pd.read_csv(history_path)

    history["gap"] = history["val_loss"] - history["train_loss"]

    summary_gap = history.groupby(["seed", "outer_fold"]).agg({
        "train_loss": "last",
        "val_loss": "last"
    }).reset_index()

    summary_gap["gap"] = summary_gap["val_loss"] - summary_gap["train_loss"]

    overfit_summary = summary_gap.groupby("seed").agg({
        "gap": ["mean", "std"]
    })

    overfit_summary.columns = ["gap_mean", "gap_std"]
    overfit_summary = overfit_summary.reset_index()

    overfit_summary["Model"] = model

    all_results.append(overfit_summary)

final_overfit = pd.concat(all_results, ignore_index=True)

final_overfit

,seed,gap_mean,gap_std,Model
0,42,0.301793,0.086989,LSTM
1,123,0.315937,0.101948,LSTM
2,2024,0.331889,0.119613,LSTM
3,42,0.085528,0.026504,CONVLSTM
4,123,0.120640,0.023259,CONVLSTM
5,2024,0.102050,0.016064,CONVLSTM
6,42,0.349910,0.155646,STGNN
7,123,0.344154,0.166403,STGNN
8,2024,0.342878,0.152486,STGNN
9,42,0.341189,0.106184,GRU


El análisis de la brecha de generalización (gap = val_loss − train_loss) muestra una separación estructural clara entre modelos en términos de sobreajuste. El Transformer presenta los valores más bajos de gap_mean (≈0.02–0.03) junto con baja variabilidad entre semillas, lo que indica una dinámica de entrenamiento bien regularizada y una capacidad de generalización consistente, sin evidencia de sobreajuste relevante. CONVLSTM también exhibe un comportamiento favorable, con gaps moderados (≈0.085–0.120) y baja dispersión relativa, lo que sugiere un equilibrio estable entre ajuste y generalización. En contraste, GRU y LSTM muestran brechas significativamente mayores (≈0.28–0.34), junto con mayor variabilidad entre semillas, lo que es consistente con sobreajuste moderado y sensibilidad a la inicialización. STGNN presenta los valores más altos y más estables del gap (≈0.34–0.35 con baja variación relativa), lo que indica un sobreajuste estructural persistente y sistemático que no depende de la semilla, reflejando una incapacidad del modelo para reducir la brecha de generalización. En conjunto, los resultados ordenan los modelos en dos regímenes: Transformer y CONVLSTM con buena generalización, y GRU/LSTM/STGNN con sobreajuste significativo, siendo STGNN el caso más crítico por su consistencia en el error de generalización.


In [20]:
model_summary = final_overfit.groupby("Model").agg({
    "gap_mean": ["mean", "std"],
    "gap_std": "mean"
})

model_summary.columns = [
    "gap_mean_mean",
    "gap_mean_std",
    "gap_std_mean"
]

model_summary = model_summary.reset_index()

print(model_summary)

         Model  gap_mean_mean  gap_mean_std  gap_std_mean
0     CONVLSTM       0.102739      0.017566      0.021942
1          GRU       0.310235      0.028894      0.080299
2         LSTM       0.316540      0.015057      0.102850
3        STGNN       0.345647      0.003746      0.158178
4  Transformer       0.024998      0.007128      0.021288


El análisis de la brecha de generalización muestra dos comportamientos bien diferenciados. El Transformer presenta el menor gap promedio (≈0.025) y baja variabilidad, indicando buena capacidad de generalización y ausencia de sobreajuste significativo. CONVLSTM se ubica en un nivel intermedio (≈0.103), con comportamiento estable y balanceado. En contraste, GRU y LSTM exhiben gaps elevados (≈0.31–0.32), consistentes con sobreajuste marcado, mientras que STGNN presenta el mayor gap (≈0.346) y baja variabilidad, lo que sugiere sobreajuste estructural persistente. En conjunto, Transformer es el mejor generalizador, seguido de CONVLSTM, mientras que GRU/LSTM/STGNN muestran generalización deficiente.

## **9.9. Conclusión**

Los resultados obtenidos a lo largo del conjunto de experimentos permiten realizar una evaluación comparativa integral del desempeño de las arquitecturas analizadas, considerando métricas de error, capacidad explicativa, estabilidad estadística y generalización.

En términos de desempeño global, el modelo Transformer presenta los mejores resultados, alcanzando los menores valores de RMSE y MAE, así como los mayores valores de R². Estos resultados indican una mayor capacidad para modelar la variabilidad del fenómeno analizado. Este comportamiento se mantiene consistente a través de diferentes semillas, evidenciando una reducción sostenida del error en comparación con los demás modelos evaluados. En contraste, las arquitecturas GRU, LSTM y STGNN exhiben un desempeño similar entre sí, caracterizado por errores elevados y valores de R² cercanos a cero o negativos, lo cual sugiere una capacidad explicativa limitada respecto a la dinámica del proceso modelado. Por su parte, CONVLSTM presenta un desempeño intermedio, con mejoras respecto a los modelos recurrentes tradicionales en términos de estabilidad y error, aunque sin alcanzar el nivel de desempeño del Transformer.

El análisis inferencial mediante ANOVA indica la existencia de diferencias estadísticamente significativas entre los modelos en todas las métricas evaluadas, lo que sugiere que las variaciones observadas no pueden atribuirse al azar. Este resultado es consistente con el análisis post-hoc de Tukey HSD, el cual evidencia una estructura de agrupación diferenciada: GRU, LSTM y STGNN conforman un grupo sin diferencias estadísticamente significativas entre sí y asociado al menor desempeño; CONVLSTM se ubica en un nivel intermedio con diferencias significativas respecto al grupo anterior; y el Transformer se diferencia significativamente de todos los demás modelos, presentando el mejor desempeño relativo.

Desde la perspectiva de la estabilidad, las métricas de variabilidad y coeficiente de variación indican que CONVLSTM presenta la menor variabilidad relativa. El Transformer, aunque exhibe mayor dispersión en comparación con CONVLSTM, mantiene consistentemente los mejores valores de error absoluto. En contraste, GRU y LSTM muestran simultáneamente alta variabilidad y bajo desempeño, lo cual sugiere sensibilidad a la inicialización y limitada robustez del proceso de aprendizaje. STGNN, si bien presenta variabilidad relativamente menor en algunas métricas, lo hace en un contexto de bajo desempeño general.

El análisis de la brecha entre entrenamiento y validación evidencia diferencias relevantes en la capacidad de generalización. El Transformer presenta la menor discrepancia entre ambas etapas, lo que indica un mejor equilibrio entre ajuste al conjunto de entrenamiento y desempeño en datos no observados. CONVLSTM presenta un comportamiento intermedio, mientras que GRU, LSTM y especialmente STGNN exhiben brechas más amplias, consistentes con posibles problemas de sobreajuste o con una representación insuficiente de la complejidad del problema.

En conjunto, los resultados experimentales y estadísticos permiten establecer una jerarquía consistente de desempeño entre los modelos evaluados. El Transformer se posiciona como el modelo con mejor desempeño global, destacándose en capacidad predictiva, diferencias estadísticamente significativas respecto a los demás modelos y adecuada generalización. CONVLSTM se ubica como una alternativa intermedia con desempeño estable pero inferior. Finalmente, GRU, LSTM y STGNN conforman un grupo de bajo desempeño sin diferencias significativas entre sí.